In [1]:
mkdir -p ./data/{01_RAW,02_CLEANED,03_TRAINING,04_VALIDATION,05_DEMO,00_MODEL}

In [15]:
import pandas as pd
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split
# INPUT RAW
raw = pd.read_csv("./data/01_RAW/IMDBDataset.csv", encoding="utf-8-sig")
raw = raw[["review", "sentiment"]]
raw = raw.drop_duplicates()

# Remove HTML tags
raw["review"] = raw["review"].apply(
    lambda x: BeautifulSoup(str(x), "html.parser").get_text()
)

# Map sentiment labels to numeric values
raw["sentiment"] = raw["sentiment"].map({
    "negative": 0,
    "positive": 2
})

# -> ./data/02_CLEANED.csv
raw.to_csv("./data/02_CLEANED/data.csv", index=False, encoding="utf-8-sig")

# -> ./data/03_TRAINING.csv
cleaned = pd.read_csv("./data/02_CLEANED/data.csv", encoding="utf-8-sig")
# Take only random 75% of sentiment 0 (negative) AND 2 (positive)
cleaned = cleaned[cleaned["sentiment"].isin([0, 2])]
negative = cleaned[cleaned["sentiment"] == 0]
positive = cleaned[cleaned["sentiment"] == 2]

n = min(len(negative), len(positive))

cleaned = pd.concat([
    negative.sample(n=n, random_state=42),
    positive.sample(n=n, random_state=42)
])


# Shuffle dataset
cleaned = cleaned.sample(frac=1, random_state=42)

# Split 75% TRAINING and 25% remaining
training, remaining = train_test_split(
    cleaned,
    test_size=0.25,
    random_state=42,
    stratify=cleaned["sentiment"]
)

# Split remaining into 24% VALIDATION and 1% DEMO
validation, demo = train_test_split(
    remaining,
    test_size=0.04,
    random_state=42,
    stratify=remaining["sentiment"]
)

# -> ./data/03_TRAINING/data.csv
training.to_csv("./data/03_TRAINING/data.csv", index=False, encoding="utf-8-sig")

# -> ./data/04_VALIDATION/data.csv
validation.to_csv("./data/04_VALIDATION/data.csv", index=False, encoding="utf-8-sig")

# -> ./data/05_DEMO/data.csv
demo.to_csv("./data/05_DEMO/data.csv", index=False, encoding="utf-8-sig")


# SUMMARY
total = len(cleaned)

print("Dataset successfully processed!")
print(f"TOTAL:      {total}")
print(f"TRAINING:   {len(training)} ({len(training) / total * 100:.2f}%)")
print(f"VALIDATION: {len(validation)} ({len(validation) / total * 100:.2f}%)")
print(f"DEMO:       {len(demo)} ({len(demo) / total * 100:.2f}%)")

print("\nSentiment distribution:")
for name, dataset in [
    ("TRAINING", training),
    ("VALIDATION", validation),
    ("DEMO", demo)
]:
    print(f"\n{name}:")
    print(dataset["sentiment"].value_counts().sort_index())

Dataset successfully processed!
TOTAL:      49396
TRAINING:   37047 (75.00%)
VALIDATION: 11855 (24.00%)
DEMO:       494 (1.00%)

Sentiment distribution:

TRAINING:
sentiment
0    18523
2    18524
Name: count, dtype: int64

VALIDATION:
sentiment
0    5928
2    5927
Name: count, dtype: int64

DEMO:
sentiment
0    247
2    247
Name: count, dtype: int64


In [18]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import Perceptron

import joblib

#
dataset = pd.read_csv("./data/03_TRAINING/data.csv", encoding="UTF-8")

# Vectorize text
vectorizer = TfidfVectorizer(
    max_features=None,
    lowercase=True,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    strip_accents="unicode"
)

# Create FEATURE
X = vectorizer.fit_transform(dataset["review"])

# Create LABEL
y = dataset["sentiment"]

# Initalize
model = Perceptron(random_state=42)
#model = LogisticRegression(random_state=42)
#model = DecisionTreeClassifier(random_state=42)

model.fit(X, y)
joblib.dump(model, "./data/00_MODEL/reviewsentiment.joblib")
joblib.dump(vectorizer, "./data/00_MODEL/vectorizer.joblib")

print(f"Successfully trained on {len(dataset)} reviews!")




validation = pd.read_csv("./data/04_VALIDATION/data.csv", encoding="utf-8-sig")
X_validation = vectorizer.transform(validation["review"])
y_validation = validation["sentiment"]

# Predict sentiment
predictions = model.predict(X_validation)

# -> VALIDATION RESULTS
accuracy = accuracy_score(y_validation, predictions)

correct = (y_validation == predictions).sum()
incorrect = (y_validation != predictions).sum()
total = len(y_validation)

print("\n========== VALIDATION RESULTS ==========")

print(f"Total reviews: {total}")
print(f"Correct predictions: {correct}")
print(f"Incorrect predictions: {incorrect}")
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\n========== SUMMARY ==========")

print(f"The model correctly classified {correct} out of {total} reviews.")
print(f"Overall accuracy: {accuracy * 100:.2f}%")


Successfully trained on 37047 reviews!

========== VALIDATION RESULTS ==========
Total reviews: 11855
Correct predictions: 10708
Incorrect predictions: 1147
Accuracy: 90.32%

========== SUMMARY ==========
The model correctly classified 10708 out of 11855 reviews.
Overall accuracy: 90.32%


Test na opiniach z datasetu przewidzianego do DEMO

In [19]:
import pandas as pd
import joblib

# Load model and vectorizer
model = joblib.load("./data/00_MODEL/reviewsentiment.joblib")
vectorizer = joblib.load("./data/00_MODEL/vectorizer.joblib")

# Load demo dataset
demo = pd.read_csv("./data/05_DEMO/data.csv", encoding="utf-8-sig")

# Vectorize reviews
X_demo = vectorizer.transform(demo["review"])

# Predict sentiment
predictions = model.predict(X_demo)

# Add predictions to dataframe
demo["predicted_sentiment"] = predictions

# Random 5 reviews
demo_sample = demo.sample(n=5, random_state=42)

for index, row in demo_sample.iterrows():
    actual = "NEGATIVE" if row["sentiment"] == 0 else "POSITIVE" if row["sentiment"] == 2 else "UNKNOWN"
    predicted = "NEGATIVE" if row["predicted_sentiment"] == 0 else "POSITIVE"

    print(f"REVIEW: {row['review']}")
    print(f"ACTUAL: {actual}")
    print(f"PREDICTED: {predicted}")
    print("-" * 80)

REVIEW: "Cypher" is a cleverly conceived story about industrial espionage set in America in the not too distant future. While thematically not complex, this film does offer many different perspectives about personal loyalty, ruthlessness, and corporate conspiracy. To a certain extent this film also attempts to represent modern corporate groups and companies as being indifferent to the risks their contract employees take on their behalf.The film starts off with a somewhat mediocre salary man, Morgan Sullivan (Jeremy Northam), who applies to the Digicorp group to work as an undercover operative. After an initial briefing with Digicorp's Security Chief, Sullivan is then given a new identity (Jack Thursby) and sent to a business conference with the task of recording the speeches given by various spokesmen concerning the marketing strategies of each of their respective companies. Upon successfully completing his first assignment, Sullivan/Thursby is sent on further missions to obtain the sa

Test na krótkich opiniach (innych niż z datasetu)

In [20]:
opinie = [
    # Positive
    "An absolutely brilliant movie with outstanding performances and a captivating story",
    "One of the best films I have watched in years, I would happily watch it again",
    "Fantastic cinematography, great acting and a story that kept me entertained throughout",
    "A wonderful movie that exceeded my expectations and left me genuinely impressed",
    "The film was incredibly entertaining, beautifully made and definitely worth watching",

    # Negative
    "A terrible movie with a boring plot and painfully bad acting",
    "I regret wasting my time watching this film, nothing about it was enjoyable",
    "The story was predictable, the characters were dull and the ending was disappointing",
    "One of the worst movies I have ever seen, completely lacking originality",
    "Poorly written, badly directed and incredibly boring from beginning to end",

    # Positive - tricky
    "The movie had a few flaws, but overall I really enjoyed watching it",
    "I expected this film to be terrible, but it turned out to be surprisingly brilliant",
    "Although the acting wasn't perfect, the story was incredibly engaging and enjoyable",
    "I can't say the movie was bad, quite the opposite, I absolutely loved it",
    "Despite its slow beginning, the film turned out to be a fantastic experience",

    # Negative - tricky
    "The movie had some great moments, but overall it was disappointing and boring",
    "It wasn't the worst film I've seen, but I certainly wouldn't recommend it",
    "Although the actors did their best, the script was terrible and the story was dull",
    "I wanted to enjoy this movie, but it failed to keep me interested",
    "The film had potential and a few good scenes, unfortunately the execution was awful"
]

# 0 - negative
# 2 - positive

vectorized_opinie = vectorizer.transform(opinie)
predictions = model.predict(vectorized_opinie)

for opinia, prediction in zip(opinie, predictions):
    etykieta = "positive" if prediction == 2 else "negative"
    print(f"'{opinia}' -> {prediction} ({etykieta})")

'An absolutely brilliant movie with outstanding performances and a captivating story' -> 2 (positive)
'One of the best films I have watched in years, I would happily watch it again' -> 2 (positive)
'Fantastic cinematography, great acting and a story that kept me entertained throughout' -> 2 (positive)
'A wonderful movie that exceeded my expectations and left me genuinely impressed' -> 2 (positive)
'The film was incredibly entertaining, beautifully made and definitely worth watching' -> 2 (positive)
'A terrible movie with a boring plot and painfully bad acting' -> 0 (negative)
'I regret wasting my time watching this film, nothing about it was enjoyable' -> 0 (negative)
'The story was predictable, the characters were dull and the ending was disappointing' -> 0 (negative)
'One of the worst movies I have ever seen, completely lacking originality' -> 0 (negative)
'Poorly written, badly directed and incredibly boring from beginning to end' -> 0 (negative)
'The movie had a few flaws, but over